# Object Detection & Feature Extraction — Classroom Practical Notebook

This notebook follows the visible topics in the supplied syllabus:
- Object detection overview
- SSD and multi-box idea
- Predicting object position and scale problem
- Feature description theory and corner detection
- SIFT, SURF, FAST, BRIEF, ORB
- Histogram of Oriented Gradients (HOG)

**Teaching approach:** first learn classical OpenCV feature methods, then run a pretrained SSD detector.

## Install
Run the next cell. If OpenCV imports fail, restart the Jupyter kernel after installation.

In [ ]:
%pip install -U opencv-python numpy matplotlib

## 1. Imports and image setup

Use your own photo or a sample image. For the best classroom results, use an image containing several clear objects.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print("OpenCV version:", cv2.__version__)

## 2. Load and display an image

Update `IMAGE_PATH` to the image you want to demonstrate. OpenCV loads images in BGR order; Matplotlib expects RGB, so convert before displaying.

In [ ]:
IMAGE_PATH = "sample.jpg"  # Change to your image file

image = cv2.imread(IMAGE_PATH)
if image is None:
    raise FileNotFoundError(
        f"Could not load {IMAGE_PATH}. Put the image beside the notebook or update IMAGE_PATH."
    )

rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(10, 6))
plt.imshow(rgb)
plt.title("Input image")
plt.axis("off")
plt.show()

## 3. Object detection overview — bounding boxes

A detector predicts:
- **Class label:** what the object is.
- **Confidence score:** how confident the model is.
- **Bounding box:** where the object is, usually represented by `(x_min, y_min, x_max, y_max)`.

This helper draws boxes so students can understand coordinates even before using a model.

In [ ]:
def draw_box(image_bgr, box, label="object", color=(0, 255, 0)):
    x1, y1, x2, y2 = map(int, box)
    output = image_bgr.copy()
    cv2.rectangle(output, (x1, y1), (x2, y2), color, 2)
    cv2.putText(
        output, label, (x1, max(25, y1 - 8)),
        cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2
    )
    return output

h, w = image.shape[:2]
example_box = (int(w * 0.15), int(h * 0.15), int(w * 0.70), int(h * 0.75))
boxed = draw_box(image, example_box, "example box")
plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(boxed, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

## 4. Predicting object position and the scale problem

A box has position and size. Normalized coordinates range approximately from 0 to 1 and are useful when image dimensions change.

The scale problem occurs because the same object may appear tiny or large depending on distance, camera, and image resolution. Classical methods and detectors can respond differently to scale changes.

In [ ]:
def box_to_normalized(box, image_width, image_height):
    x1, y1, x2, y2 = box
    return (
        x1 / image_width,
        y1 / image_height,
        x2 / image_width,
        y2 / image_height,
    )

print("Pixel box:", example_box)
print("Normalized box:", box_to_normalized(example_box, w, h))

## 5. Convert to grayscale

Many classical feature detectors work on grayscale intensity patterns.

In [ ]:
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
plt.figure(figsize=(10, 5))
plt.imshow(gray, cmap="gray")
plt.title("Grayscale image")
plt.axis("off")
plt.show()

## 6. Corner detection with Shi–Tomasi

Corners are locations where image intensity changes strongly in two directions. They are useful for tracking and matching visual features.

In [ ]:
corners = cv2.goodFeaturesToTrack(
    gray,
    maxCorners=150,
    qualityLevel=0.01,
    minDistance=8
)

corner_vis = image.copy()
if corners is not None:
    corners = corners.astype(np.int32)
    for corner in corners:
        x, y = corner.ravel()
        cv2.circle(corner_vis, (x, y), 4, (0, 0, 255), -1)

plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(corner_vis, cv2.COLOR_BGR2RGB))
plt.title("Shi–Tomasi corners")
plt.axis("off")
plt.show()

## 7. FAST keypoint detector

FAST quickly identifies candidate corners. It detects keypoints but does not itself compute a full descriptor for matching.

In [ ]:
fast = cv2.FastFeatureDetector_create()
fast_keypoints = fast.detect(gray, None)
fast_vis = cv2.drawKeypoints(
    image, fast_keypoints, None,
    color=(0, 255, 0),
    flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
)
print("FAST keypoints:", len(fast_keypoints))
plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(fast_vis, cv2.COLOR_BGR2RGB))
plt.title("FAST keypoints")
plt.axis("off")
plt.show()

## 8. ORB — FAST keypoints + BRIEF-like binary descriptors

ORB (Oriented FAST and Rotated BRIEF) is a popular free alternative for keypoint detection and binary feature description. It is often useful for real-time feature matching.

In [ ]:
orb = cv2.ORB_create(nfeatures=500)
orb_keypoints, orb_descriptors = orb.detectAndCompute(gray, None)
orb_vis = cv2.drawKeypoints(
    image, orb_keypoints, None,
    color=(255, 0, 0),
    flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
)
print("ORB keypoints:", len(orb_keypoints))
print("Descriptor shape:", None if orb_descriptors is None else orb_descriptors.shape)
plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(orb_vis, cv2.COLOR_BGR2RGB))
plt.title("ORB keypoints")
plt.axis("off")
plt.show()

## 9. SIFT descriptors

SIFT finds scale-space keypoints and computes descriptors that are more robust to scale and rotation changes than simple corner-only methods. Availability depends on the OpenCV build.

In [ ]:
if hasattr(cv2, "SIFT_create"):
    sift = cv2.SIFT_create()
    sift_keypoints, sift_descriptors = sift.detectAndCompute(gray, None)
    sift_vis = cv2.drawKeypoints(
        image, sift_keypoints, None,
        flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
    )
    print("SIFT keypoints:", len(sift_keypoints))
    print("SIFT descriptor shape:", None if sift_descriptors is None else sift_descriptors.shape)
    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(sift_vis, cv2.COLOR_BGR2RGB))
    plt.title("SIFT keypoints")
    plt.axis("off")
    plt.show()
else:
    print("SIFT is not available in this OpenCV build.")

## 10. SURF note

SURF is a scale- and rotation-aware feature detector/descriptor. In many OpenCV distributions it is not enabled because it belongs to the non-free/contrib area. Do not assume `cv2.xfeatures2d.SURF_create()` exists in every installation.

In [ ]:
if hasattr(cv2, "xfeatures2d") and hasattr(cv2.xfeatures2d, "SURF_create"):
    try:
        surf = cv2.xfeatures2d.SURF_create()
        surf_keypoints, surf_descriptors = surf.detectAndCompute(gray, None)
        print("SURF keypoints:", len(surf_keypoints))
    except cv2.error as exc:
        print("SURF exists but is unavailable in this build/configuration:", exc)
else:
    print("SURF is unavailable in this OpenCV installation. Use SIFT or ORB for the practical.")

## 11. BRIEF concept

BRIEF creates a binary descriptor by comparing pairs of pixel intensities around a keypoint. OpenCV Python builds often do not expose a standalone BRIEF constructor; ORB includes a rotated BRIEF-like descriptor.

In [ ]:
print("ORB descriptor dtype:", None if orb_descriptors is None else orb_descriptors.dtype)
print("ORB descriptor shape:", None if orb_descriptors is None else orb_descriptors.shape)
print("Binary descriptors are commonly matched with Hamming distance.")

## 12. Histogram of Oriented Gradients (HOG)

HOG summarizes local edge directions and is a classical feature representation used in object detection, including pedestrian detection.

In [ ]:
hog = cv2.HOGDescriptor()
hog_vis = image.copy()
print("HOG descriptor window size:", hog.winSize)
print("HOG block size:", hog.blockSize)
print("HOG cell size:", hog.cellSize)
print("HOG block stride:", hog.blockStride)

## 13. HOG pedestrian detector (optional)

OpenCV ships a default people detector for HOG. It works best on full-body pedestrian images and is not a general detector for every object category.

In [ ]:
hog.setSVMDetector(cv2.HOGDescriptor_getDefaultPeopleDetector())
rectangles, weights = hog.detectMultiScale(
    image,
    winStride=(8, 8),
    padding=(8, 8),
    scale=1.05
)
hog_result = image.copy()
for (x, y, bw, bh), weight in zip(rectangles, weights):
    cv2.rectangle(hog_result, (x, y), (x + bw, y + bh), (0, 255, 0), 2)
    cv2.putText(hog_result, f"person {float(weight):.2f}", (x, max(20, y-5)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0,255,0), 2)
print("Pedestrian detections:", len(rectangles))
plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(hog_result, cv2.COLOR_BGR2RGB))
plt.title("HOG + SVM pedestrian detection")
plt.axis("off")
plt.show()

## 14. SSD object detector — pretrained model

SSD (Single Shot MultiBox Detector) predicts class scores and box offsets from multiple feature maps in a single forward pass. The model below uses OpenCV's DNN module and a pretrained Caffe SSD model.

Download these two model files from the official OpenCV Model Zoo / OpenCV samples repository and put them in a folder named `models`:
- `MobileNetSSD_deploy.prototxt`
- `MobileNetSSD_deploy.caffemodel`

If you do not have these model files, skip the SSD cell for now. The classical OpenCV sections above can still run.

In [ ]:
from pathlib import Path

PROTOTXT = Path("models/MobileNetSSD_deploy.prototxt")
CAFFEMODEL = Path("models/MobileNetSSD_deploy.caffemodel")

if not PROTOTXT.exists() or not CAFFEMODEL.exists():
    print("SSD model files not found. Put both files in ./models to run SSD inference.")
else:
    CLASSES = [
        "background", "aeroplane", "bicycle", "bird", "boat", "bottle",
        "bus", "car", "cat", "chair", "cow", "diningtable", "dog", "horse",
        "motorbike", "person", "pottedplant", "sheep", "sofa", "train", "tvmonitor"
    ]
    net = cv2.dnn.readNetFromCaffe(str(PROTOTXT), str(CAFFEMODEL))
    blob = cv2.dnn.blobFromImage(
        cv2.resize(image, (300, 300)),
        scalefactor=0.007843,
        size=(300, 300),
        mean=127.5
    )
    net.setInput(blob)
    detections = net.forward()
    output = image.copy()
    confidence_threshold = 0.45
    height, width = output.shape[:2]

    for i in range(detections.shape[2]):
        confidence = float(detections[0, 0, i, 2])
        class_id = int(detections[0, 0, i, 1])
        if confidence < confidence_threshold:
            continue
        x1, y1, x2, y2 = (detections[0, 0, i, 3:7] * np.array([width, height, width, height])).astype(int)
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(width-1, x2), min(height-1, y2)
        label = CLASSES[class_id] if 0 <= class_id < len(CLASSES) else str(class_id)
        cv2.rectangle(output, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(output, f"{label}: {confidence:.2f}", (x1, max(20, y1-5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0,255,0), 2)

    plt.figure(figsize=(12, 7))
    plt.imshow(cv2.cvtColor(output, cv2.COLOR_BGR2RGB))
    plt.title("SSD detections")
    plt.axis("off")
    plt.show()

## 15. The MultiBox concept and non-maximum suppression (NMS)

SSD evaluates many candidate boxes (often called default boxes/anchors) at different scales and aspect ratios. The model predicts class scores and offsets for these boxes. Multiple candidates can overlap around the same object, so non-maximum suppression (NMS) helps keep the strongest box and remove redundant overlaps.

This simplified example demonstrates NMS independently of SSD.

In [ ]:
candidate_boxes = [
    [40, 50, 180, 200],
    [45, 55, 178, 198],
    [220, 70, 350, 210],
]
scores = [0.92, 0.78, 0.88]
# OpenCV NMSBoxes expects boxes as (x, y, width, height).
xywh_boxes = [[x1, y1, x2-x1, y2-y1] for x1, y1, x2, y2 in candidate_boxes]
keep = cv2.dnn.NMSBoxes(xywh_boxes, scores, score_threshold=0.3, nms_threshold=0.4)
print("Kept candidate indices:", keep)

## 16. SIFT/ORB feature matching (optional)

This section matches features between two images. Provide `second.jpg` beside the notebook.

In [ ]:
SECOND_IMAGE_PATH = "second.jpg"
second = cv2.imread(SECOND_IMAGE_PATH)

if second is None:
    print("Put a second image named second.jpg beside the notebook to run feature matching.")
else:
    gray2 = cv2.cvtColor(second, cv2.COLOR_BGR2GRAY)
    orb = cv2.ORB_create(nfeatures=800)
    kp1, des1 = orb.detectAndCompute(gray, None)
    kp2, des2 = orb.detectAndCompute(gray2, None)

    if des1 is None or des2 is None:
        print("Not enough ORB descriptors in one of the images.")
    else:
        matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
        matches = sorted(matcher.match(des1, des2), key=lambda m: m.distance)[:40]
        matched = cv2.drawMatches(image, kp1, second, kp2, matches, None,
                                  flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
        plt.figure(figsize=(14, 7))
        plt.imshow(cv2.cvtColor(matched, cv2.COLOR_BGR2RGB))
        plt.title("ORB feature matches")
        plt.axis("off")
        plt.show()

## 17. Student exercises

1. Compare FAST, ORB, and SIFT keypoint counts on the same image.
2. Test the same object at two different sizes to discuss the scale problem.
3. Run HOG on an image with people and observe false positives/negatives.
4. Run SSD with confidence thresholds 0.25, 0.45, and 0.65.
5. Explain why SSD can predict multiple boxes for one object and why NMS is useful.
6. Compare ORB's binary descriptors with SIFT's floating-point descriptors.

## Important teaching notes
- Classical feature detectors/descriptors do not by themselves provide general semantic object labels.
- HOG + SVM pedestrian detection is a specialized classical detector, not a general-purpose detector.
- SSD requires compatible pretrained model files; the notebook does not download them automatically.